# Does an Inverted Yield Curve Reliably Predict Recessions?

**Aidan Braccia — ECON 630, Assignment 6**

Question, data sources, and method are laid out in `Plan.md`. This notebook
follows that plan stage by stage.

## Stage 1 — Data pull, validation, and cleaning

Everything is pulled live from FRED on each run, so no CSVs are stored in the
repo. A spot check on the spread runs before any analysis, so a wrong sign or
bad series fails loudly instead of quietly corrupting downstream results.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred

load_dotenv()                                   # reads FRED_API_KEY from .env (gitignored)
fred = Fred(api_key=os.getenv("FRED_API_KEY"))  # key is never typed into the notebook

### Pull the three series

Each `get_series` call returns a plain `pd.Series` with a `DatetimeIndex` and no
column name — one value per month.

In [ ]:
gs10  = fred.get_series("GS10")   # 10-Year Treasury Constant Maturity Rate, monthly, percent
gs2   = fred.get_series("GS2")    # 2-Year  Treasury Constant Maturity Rate, monthly, percent
usrec = fred.get_series("USREC")  # NBER recession indicator, monthly, 1 = recession

for name, s in [("GS10", gs10), ("GS2", gs2), ("USREC", usrec)]:
    print(f"{name:6} {len(s):5} observations   {s.index.min():%Y-%m} to {s.index.max():%Y-%m}")

### Trim to the sample period and build the spread

The sample starts in 1976-06, the first month of FRED's `GS2` series, so this is
the longest window the data allows. It covers three monetary policy regimes
(the late-1970s inflation, the Volcker disinflation, and the post-1990 era), six
NBER recessions (1980, 1981–82, 1990–91, 2001, 2007–09, 2020) and the 2022–24
inversion episode.

`.loc[START:]` keeps every month from June 1976 onward. It is a slice from that
date to the end of the series, not a match on that single date.

The sign is the thing to get right: `spread = GS10 - GS2`, so an inversion is
`spread < 0`. Flipping it would silently invert the entire story.

In [ ]:
START = "1976-06-01"   # first month of GS2
gs10  = gs10.loc[START:]
gs2   = gs2.loc[START:]
usrec = usrec.loc[START:]

spread = gs10 - gs2   # 10y minus 2y, so inversion is spread < 0

print(f"sample: {spread.index.min():%Y-%m} to {spread.index.max():%Y-%m}  ({len(spread)} months)")

### Validation check

The curve is known to have been inverted in **November 2006** (−0.14), ahead of
the 2007–09 recession, so the spread that month must be negative. This check
fails if the spread sign is flipped.

In [ ]:
assert spread.loc["2006-11"].item() < 0, \
    "expected the curve to be inverted in 2006-11, ahead of the 2007-2009 recession"

print(f"spread range: {spread.min():.2f} to {spread.max():.2f} percentage points")

## Stage 2 — Charts

Three charts, as laid out in `Plan.md`:

1. the spread over time with recession shading,
2. each inversion episode against the recession that followed it,
3. the lead times side by side.

All three share one set of colors so they read as a single system: blue for the
series, red reserved for inversion, neutral gray for recession context and
chrome. Blue and red are a diverging pair — they encode the polarity of the
spread (above or below zero), not two arbitrary categories.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SURFACE = "#fcfcfb"   # chart surface
INK     = "#0b0b0b"   # primary text
INK2    = "#52514e"   # secondary text
MUTED   = "#898781"   # axis labels
GRID    = "#e1e0d9"   # hairline gridlines
AXIS    = "#c3c2b7"   # baseline, recession bands
BLUE    = "#2a78d6"   # the spread series
RED     = "#e34948"   # inversion


def style(ax):
    """Shared chart chrome: recessive grid, no top/right spines, muted ticks."""
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_lw(0.8)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)


def source(fig):
    fig.text(0.005, -0.02,
             "Source: FRED (GS10, GS2, USREC). Monthly, "
             f"{spread.index[0]:%Y-%m} to {spread.index[-1]:%Y-%m}.",
             color=MUTED, fontsize=8, ha="left", va="top")

### Defining an inversion episode

The spread crosses zero more often than there are distinct inversions. The
2006–07 episode dips below zero, pops barely positive for a month or two, and
dips again. Treating each crossing as its own episode would invent inversions
that nobody would describe as separate.

Runs of `spread < 0` separated by a gap of **three months or less** are merged
into one episode (`MAX_GAP = 3`, from `Plan.md`). Single-month dips are kept
rather than filtered out: 1990-03 and 1998-06 are both one-month dips.

### Assigning a lead time to each recession

Lead times are counted **per recession, not per episode**, using the rule in
`Plan.md`. For each recession, take the most recent episode that started in the
**`SIGNAL_WINDOW` months before it**, whether or not that episode is still going
when the recession starts. The lead time runs from the episode's first month to
the recession's first month. A recession with no episode in that window is
recorded as **no inversion**.

An episode is a **false alarm** if no recession begins within **24 months of its
onset** (`SIGNAL_WINDOW = 24`, from `Plan.md`). False alarms are left out when
lead times are assigned, so a later recession can never be matched to one. The
window sits just above the longest real lead in the sample (23 months), so it is
a choice.

Each episode then gets one of three roles:
- **signal:** it produced a recession's lead time;
- **superseded:** a later episode came before the same recession;
- **false alarm:** no recession within `SIGNAL_WINDOW` months of its onset.

In [ ]:
MAX_GAP       = 3    # months of non-inversion tolerated inside a single episode
SIGNAL_WINDOW = 24   # an episode with no recession this many months after onset is a false alarm

inverted = spread < 0
run_id   = (inverted != inverted.shift()).cumsum()
runs     = [(g.index[0], g.index[-1]) for _, g in inverted[inverted].groupby(run_id[inverted])]

episodes = []
for start, end in runs:
    if episodes:
        prev_end = episodes[-1][1]
        gap = (start.year - prev_end.year) * 12 + (start.month - prev_end.month) - 1
        if gap <= MAX_GAP:
            episodes[-1] = (episodes[-1][0], end)
            continue
    episodes.append((start, end))

ep = pd.DataFrame([{"start": s, "end": e, "min_spread": spread.loc[s:e].min()}
                   for s, e in episodes])

# recession onsets: the month USREC flips 0 -> 1
onsets = list(usrec.index[(usrec == 1) & (usrec.shift(1) == 0)])


def months_between(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)


LAST = spread.index[-1]


def next_recession(start):
    later = [d for d in onsets if d > start]
    return later[0] if later else pd.NaT


# false alarm: no recession within SIGNAL_WINDOW months of onset (Plan.md)
ep["next_recession"] = ep["start"].map(next_recession)
ep["false_alarm"] = [
    (months_between(s, r) > SIGNAL_WINDOW) if pd.notna(r) else (months_between(s, LAST) > SIGNAL_WINDOW)
    for s, r in zip(ep["start"], ep["next_recession"])
]

# one row per recession: the most recent non-false-alarm episode that started in the
# SIGNAL_WINDOW months before it (Plan.md) - the same window as the false-alarm rule
rows, used = [], {}
for onset in onsets:
    window_start = onset - pd.DateOffset(months=SIGNAL_WINDOW)
    before = ep[(ep["start"] < onset) & (ep["start"] >= window_start) & ~ep["false_alarm"]]
    if before.empty:
        rows.append({"recession": onset, "stretch": pd.NaT, "lead": None, "status": "no inversion"})
        continue
    stretch = before["start"].iloc[-1]
    used[stretch] = onset
    rows.append({"recession": onset, "stretch": stretch,
                 "lead": months_between(stretch, onset), "status": "bar"})
leads = pd.DataFrame(rows).astype({"lead": "Int64"})   # whole months; blank for no bar

ep["recession"] = ep["start"].map(used)
ep["role"] = ["signal" if s in used else "false alarm" if fa
              else "open" if pd.isna(r) else "superseded"
              for s, fa, r in zip(ep["start"], ep["false_alarm"], ep["next_recession"])]
# for a superseded episode, the later episode that set the next recession's lead
ep["superseded_by"] = [leads.loc[leads["recession"] > s, "stretch"].iloc[0] if role == "superseded" else pd.NaT
                       for s, role in zip(ep["start"], ep["role"])]

print(ep.to_string(index=False))
print()
print(leads.to_string(index=False))

### Chart 1 — The spread over time

The fill is split at zero: blue above, red below. A single series needs no
legend, so the legend identifies only the two shaded encodings.

The deepest inversions in the sample are the 1978–82 ones, during the Volcker
disinflation. They also continue *into* the 1980 and 1981–82 recessions, where
every later inversion ended before its recession began. The 2020 recession band
is **two months wide** and easy to mistake for a gridline at this scale.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4), dpi=130, facecolor=SURFACE)
style(ax)

ax.fill_between(usrec.index, 0, 1, where=usrec == 1, transform=ax.get_xaxis_transform(),
                color=AXIS, alpha=.55, lw=0, zorder=1)
ax.fill_between(spread.index, spread, 0, where=spread >= 0, color=BLUE, alpha=.16, lw=0, zorder=2)
ax.fill_between(spread.index, spread, 0, where=spread < 0,  color=RED,  alpha=.30, lw=0, zorder=2)
ax.axhline(0, color=INK2, lw=1, zorder=3)
ax.plot(spread.index, spread, color=BLUE, lw=2, zorder=4)

ax.set_ylabel("Percentage points", color=INK2, fontsize=9.5)
ax.set_title(f"The 10-year minus 2-year Treasury spread, {spread.index[0]:%Y}–{LAST:%Y}",
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015, "Below zero the yield curve is inverted. Shaded bands are NBER recessions.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
ax.legend(handles=[Patch(facecolor=AXIS, alpha=.55, label="NBER recession"),
                   Patch(facecolor=RED, alpha=.30, label="Inverted (spread < 0)")],
          loc="upper right", frameon=False, fontsize=9, labelcolor=INK2)
source(fig)
plt.show()

### Chart 2 — Every episode, and the recession it was matched to

One row per episode, on a calendar axis. The bar spans the inverted months. A
solid arrow runs from the onset of each **signal** episode to the recession it
was matched to, and that arrow is the lead time in chart 3. **Superseded**
episodes are drawn faded, with no arrow, because a later episode came before the
same recession. **False alarms** (1998-06 and 2022-07) get no
arrow and are labeled with how long they went without a recession.

Two cases stand out: the one-month 1990-03 dip that sets the 1990–91 lead, and
the 2020 recession, which had no inversion in the 24 months before it. The
most recent inversion before 2020 began in 2006-02, **169 months** earlier. That
is more than seven times the longest real lead time, so it is not treated as a
signal. The 2022 inversion is
the deepest since the early 1980s (−0.93), and it is also a false alarm.

In [ ]:
# contiguous recession spans, for shading
rec_id = (usrec != usrec.shift()).cumsum()
rec_spans = [(b.index[0], b.index[-1] + pd.DateOffset(months=1))
             for _, b in usrec[usrec == 1].groupby(rec_id[usrec == 1])]

fig, ax = plt.subplots(figsize=(11, 5.0), dpi=130, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for s0, e0 in rec_spans:
    ax.axvspan(s0, e0, color=AXIS, alpha=.55, lw=0, zorder=1)
ax.grid(axis="x", color=GRID, lw=.8, zorder=0)
ax.set_axisbelow(True)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color(AXIS); ax.spines["bottom"].set_lw(.8)
ax.tick_params(colors=MUTED, labelsize=9, length=0)

y_rows = list(range(len(ep)))[::-1]          # earliest episode on top
for i, (_, r) in zip(y_rows, ep.iterrows()):
    faded = r.role == "superseded"
    ax.barh(i, (r.end + pd.DateOffset(months=1)) - r.start, left=r.start,
            height=.34, color=RED, alpha=.3 if faded else .85, lw=0, zorder=3)
    # onset marker, so one-month episodes (1990-03, 1998-06) stay visible at this scale
    ax.plot(r.start, i, marker="o", ms=5, color=RED, alpha=.45 if faded else 1,
            mec=SURFACE, mew=1, zorder=4)
    if faded:
        ax.text(r.end + pd.DateOffset(months=8), i, f"superseded by {r.superseded_by:%b %Y}",
                va="center", fontsize=8.5, color=MUTED, zorder=5)
        continue
    if r.role == "false alarm":
        waited = months_between(r.start, LAST if pd.isna(r.next_recession) else r.next_recession)
        text = ("false alarm: no recession in " if pd.isna(r.next_recession)
                else "false alarm: next recession ") + f"{waited} mo" + ("" if pd.isna(r.next_recession) else " later")
        ax.text(r.end + pd.DateOffset(months=8), i, text,
                va="center", fontsize=9, fontweight="bold", color=RED, zorder=5)
        continue
    lead = months_between(r.start, r.recession)
    ax.annotate("", xy=(r.recession, i + .40), xytext=(r.start, i + .40),
                arrowprops=dict(arrowstyle="->", color=INK2, lw=1.2, shrinkA=0, shrinkB=0),
                zorder=4)
    ax.text(max(r.recession, r.end + pd.DateOffset(months=1)) + pd.DateOffset(months=6), i + .15,
            f"{lead} mo", ha="left", va="center", fontsize=9, fontweight="bold",
            color=INK, zorder=5)

ax.set_yticks(y_rows)
ax.set_yticklabels([f"{r.start:%b %Y}" for _, r in ep.iterrows()], color=INK2, fontsize=10)
ax.set_ylim(-.7, len(ep) - .25)
ax.set_xlim(spread.index[0] - pd.DateOffset(months=6), LAST + pd.DateOffset(months=60))
ax.set_title("Each inversion episode and the recession it was matched to",
             color=INK, fontsize=13, fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015,
        "Solid arrows run from a signal episode's onset to its recession. Faded bars were superseded; false alarms get no arrow.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
ax.legend(handles=[Patch(facecolor=RED, alpha=.85, label="Inversion episode"),
                   Patch(facecolor=AXIS, alpha=.55, label="NBER recession")],
          loc="lower left", frameon=False, fontsize=9, labelcolor=INK2, ncol=2)
source(fig)
plt.show()

### Chart 3 — Lead times, one per recession

The plan called this the chart that decides the question: a consistent signal
produces bars of similar length, a noisy one produces scatter. There is one row
per recession. The 2020 row has no bar because no inversion started in the 24
months before it.

The two false alarms, 1998-06 and 2022-07, are drawn open and hatched rather
than filled, and they sit below the recessions because they are not recessions.
Each bar runs from the episode's onset to the next recession or, for 2022, to
the end of the data. Both run past the dashed 24-month line (`SIGNAL_WINDOW`),
and that is what makes them false alarms. They carry their own labels, so the
distinction never rests on the hatching alone.

In [ ]:
false_alarms = ep[ep["role"] == "false alarm"]
chart_rows = [(f"{r.recession:%Y} recession", r.lead, r.status, r.stretch)
              for _, r in leads.iterrows()]
chart_rows += [(f"{r.start:%Y} inversion",
                months_between(r.start, LAST if pd.isna(r.next_recession) else r.next_recession),
                "false alarm", r.next_recession)
               for _, r in false_alarms.iterrows()]

fig, ax = plt.subplots(figsize=(8.6, 4.4), dpi=130, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.grid(axis="x", color=GRID, lw=0.8)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(AXIS); ax.spines["bottom"].set_lw(0.8)
ax.tick_params(colors=MUTED, labelsize=9, length=0)

y_rows = list(range(len(chart_rows)))[::-1]
for i, (label, lead, status, stretch) in zip(y_rows, chart_rows):
    if status == "no inversion":
        ax.text(1.2, i, f"no inversion in the {SIGNAL_WINDOW} months before it",
                va="center", ha="left", fontsize=9.5, color=INK2, style="italic",
                zorder=3, bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
        continue
    alarm = status == "false alarm"
    ax.barh(i, lead, height=.58, zorder=2,
            color="none" if alarm else BLUE,
            edgecolor=RED if alarm else "none",
            lw=1.6, hatch="///" if alarm else None)
    if alarm:
        label = (f"{lead} months, no recession: false alarm" if pd.isna(stretch)
                 else f"{lead} months to next recession: false alarm")
    else:
        label = f"{lead} months (from {stretch:%Y-%m})"
    ax.text(lead + 1.2, i, label, va="center", ha="left", fontsize=9.5,
            fontweight="bold", color=RED if alarm else INK, zorder=3,
            bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))

ax.axvline(SIGNAL_WINDOW, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
ax.set_ylim(-1.15, len(chart_rows) - .4)
ax.text(SIGNAL_WINDOW + .6, -.95, f"{SIGNAL_WINDOW}-month false-alarm window",
        fontsize=8.5, color=MUTED, va="center", ha="left")

ax.set_yticks(y_rows)
ax.set_yticklabels([row[0] for row in chart_rows], color=INK2, fontsize=10)
ax.set_xlim(0, 80)
ax.set_xlabel("Months from inversion onset to recession onset", color=INK2, fontsize=9.5)
ax.set_title("Lead times are not consistent", color=INK, fontsize=13,
             fontweight="bold", loc="left", pad=30)
ax.text(0, 1.015, "For each recession, months from the onset of the most recent inversion episode. Hatched rows are false alarms.",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
source(fig)
plt.show()

## Stage 3 — Statistical test

One test from `Plan.md`: a logistic regression of recession status on the
spread twelve months earlier, fit once with autocorrelation-robust (HAC)
standard errors.

First, the remaining cleaning step. `inverted` was already built in Stage 2;
`spread_lag12` is new. The regression frame drops only the twelve leading
months that the lag leaves empty.

In [ ]:
spread_lag12 = spread.shift(12)  # the spread as it stood 12 months before each date

data = pd.DataFrame({"usrec": usrec, "spread_lag12": spread_lag12}).dropna()  # drops the 12 empty lag rows

print(f"inverted months: {inverted.sum()} of {len(inverted)}")
print(f"observations: {len(data)}   recession months: {int(data['usrec'].sum())}"
      f"  ({data['usrec'].mean():.1%} of the sample)")

### Logistic regression with HAC standard errors

`USREC ~ spread_lag12`, one lag only. This goes past what has been lectured —
`statsmodels` appeared in the toolkit overview but was not taught — so each
piece is spelled out.

- `sm.add_constant` adds the intercept column. Without it the model is forced
  through zero, which silently changes what the coefficient means.
- `Logit(y, X).fit()` estimates by maximum likelihood.
- `cov_type="HAC", cov_kwds={"maxlags": 12}` gives Newey-West standard errors
  with a 12-month bandwidth.
- The coefficient is on the **log-odds** scale, so `np.exp(coef)` converts it to
  an odds ratio per one-percentage-point change in the spread.

The standard errors are HAC because recession months come in clumps, six
contiguous runs in this sample. Ordinary standard errors would treat those few
episodes as hundreds of independent observations and overstate the precision.

In [ ]:
import numpy as np
import statsmodels.api as sm

y = data["usrec"]
X = sm.add_constant(data[["spread_lag12"]])   # intercept + one regressor

logit = sm.Logit(y, X).fit(disp=0, cov_type="HAC", cov_kwds={"maxlags": 12})
print(logit.summary())

coef = logit.params["spread_lag12"]
print(f"\ncoefficient on spread_lag12: {coef:.3f}")
print(f"odds ratio per +1pp of spread: {np.exp(coef):.3f}")
print(f"i.e. each extra percentage point of spread 12 months ago multiplies the")
print(f"odds of being in recession by about {np.exp(coef):.2f}.")

The coefficient is about **−1.50**, an odds ratio of about **0.22** per
percentage point: a flatter or inverted curve a year earlier goes with higher
odds of recession now. With HAC standard errors, p = **0.008**, significant at
the 1% level.

Two cautions keep this from being the whole story:
- Six recessions is still a small sample, as `Plan.md` warns up front. This is
  a compact description of the pattern the charts show, not a well-powered
  test.
- The regression says the spread *level* a year earlier is associated with
  recession. It says nothing about *timing*. Chart 3 shows lead times from
  5 to 23 months, plus two false alarms, and 2020 had no inversion in the 24
  months before it.

---

## Conclusion

**The question:** does a negative 10-year minus 2-year spread reliably precede
U.S. recessions, or is it noisier than its reputation suggests?

**On this evidence, both, depending on what "reliable" means.** The inversion is
a real signal *that a recession is likely*. It is a poor signal of *when*, and
it sometimes fires when no recession follows. My expectation going in was that
inversions would precede most recessions, but with an inconsistent lag. That is
close to what the data shows, and the regression came out stronger than I
expected.

**What holds up:**
- **The association is strong.** In the logistic regression, the coefficient on
  the spread twelve months earlier is −1.50, an odds ratio of 0.22 per
  percentage point, and with HAC standard errors p = 0.008.
- **Most recessions were preceded by an inversion.** Five of the six recessions
  since 1976 had an inversion episode start within the 24 months before them.

**What doesn't:**
1. **Lead times do not cluster.** They were 17, 11, 5, 14 and 23 months.
   Knowing an inversion has begun tells you little about when the recession
   will arrive. Even the 5 is fragile: it comes from a one-month dip of −0.04 in
   1990-03.
2. **It produces false alarms.** The 1998-06 dip was followed by a recession
   only 34 months later. The 2022–24 inversion, the deepest since the early
   1980s (−0.93), has gone 49 months with no recession.
3. **It missed a recession.** No inversion began in the 24 months before the
   2020 recession. A pandemic is not the kind of shock the yield curve prices,
   so this is a limit on what the signal can see.

**Caveats.** There are only six recessions in the sample, so the regression is
a compact description of the pattern in the charts, not a well-powered test.
And the 24-month window is a choice, not a finding: it sits just above the
longest lead (23 months), and it decides which episodes count as false alarms.

The honest summary is that the yield curve is a reliable indicator of elevated
recession risk, and an unreliable one for forecasting when a recession will
start.

---

## How I used Claude Code

I wrote `Plan.md` before any code existed, then worked through the notebook one
stage at a time and checked each stage against the plan. That caught real
errors. The spread was once built backwards, and my own spot check on
September 2007 failed because that month was not actually inverted, so the
check moved to November 2006. The lead-time rule needed corrections too: its
first wording ("the stretch still running when the recession began") would have
left half the recessions without a lead, and the 24-month window had to run
backward from each recession as well as forward from each inversion. The final
claim is mixed: the inversion signals elevated recession risk, but not when the
recession will start.